In [27]:
import os
import sys
import json
import pickle
import os.path as osp
import random

from tqdm import tqdm
import torch
import numpy as np
import trimesh
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist

from transforms3d.axangles import axangle2mat, mat2axangle
import plotly.graph_objects as go
from mano_pybullet.hand_model import HandModel20

In [ ]:
# NOTE: Set the mano hand models dir here. When using with a script, load this directory from a some config file

%env MANO_MODELS_DIR=/home/ninad/Projects/MANO/MANO_Hand_Model/mano_v1_2/models

In [25]:
sys.path.append("..")

from utils.grasp_utils import get_handmodel
from model.hand_opt import AdamGraspTransfer
from utils.fig_utils import (
    plot_point_cloud_cmap,
    plot_uv_map,
    plot_trimesh_mesh,
    get_object_mesh,
    spherical_to_cart_coords,
    hsv_uv_colors,
    cmap_colors,   
)

In [26]:
def mat2rvec(mat):
    """Convert rotation matrix to rotation vector."""
    axis, angle = mat2axangle(mat, unit_thresh=1e-05)
    return axis * angle

def rvec2mat(rvec):
    """Convert rotation vector to rotation matrix."""
    angle = np.linalg.norm(rvec)
    axis = rvec if angle != 0.0 else [0.0, 0.0, 1.0]
    mat = axangle2mat(axis, angle)
    return mat

# Paths

In [59]:
FIG_SAVE_DIR = "./viz_results/"

source_gripper = "mano_right"
target_gripper = "fetch_gripper"
device = "cpu"

source_model = get_handmodel(
  source_gripper,
  1,
  device,
  json_path="urdf_assets_meta.json",
  datadir="../grippers/"
)

# Mano PYB Right Hand Model
hand_model = HandModel20(left_hand=False)

########## FETCH GRIPPER #############
target_model = get_handmodel(
  target_gripper,
  1,
  device,
  json_path="urdf_assets_meta.json",
  datadir="../grippers/"
)



# Load Sample Hamer Data

Sourced from newCamK -> Task 14 (Pouring) -> Frame 039
 

In [ ]:
use_left_hand = False
frame_id = "000039"
fname = f"{frame_id}.npz"
data = np.load(f"../data/{fname}", allow_pickle=True)
print("Use left hand? -->", use_left_hand)

rl_index = data['right']
left_idxs = np.arange(rl_index.shape[0])[rl_index==0]
right_idxs = np.arange(rl_index.shape[0])[rl_index==1]

print("LEFT IDXS:", left_idxs)
print("RIGHT IDXS:", right_idxs)

idx_to_use = left_idxs if use_left_hand else right_idxs
print("IDX TO USE:", idx_to_use)

In [35]:
mano_params = data['pred_mano_params'].item()
hand_rotn_mat = mano_params['global_orient'][idx_to_use][0][0]
hand_theta_mat = mano_params['hand_pose'][idx_to_use][0]
mano_trans = data['opt_translation'][idx_to_use][0]

In [36]:
# Convert MANO Params to Joint Angles and Orientation

hand_theta_full = np.array([mat2rvec(hand_rotn_mat)] + [mat2rvec(hand_theta_mat[i]) for i in range(hand_theta_mat.shape[0])])
angles, palm_basis = hand_model.mano_to_angles(hand_theta_full)

# Reference: https://github.com/kninad/mano_pybullet/blob/960c257cf465f8966e770562b66150beaa359230/mano_pybullet/hand_body.py#L155
origin = hand_model.origins()[0]
palm_trans = mano_trans + origin - palm_basis @ origin

actual_trans = np.array(palm_trans)
actual_basis = palm_basis
# actual changes if using left hand: refer to `transfer_from_hamer.ipynb` for this!


# Src Grp Pose + Transfer 

In [ ]:

########## Setup Source Gripper Pose for Transfer #############

grasp_pose = torch.zeros(9)
grasp_pose[3:] = torch.tensor(actual_basis.T.reshape(-1)[:6])
grasp_pose[:3] = torch.tensor(actual_trans)
print("Src Gripper (MANO) Pose:", grasp_pose)

grasp_dofs = torch.tensor(angles)
print("Src Gripper Angles:", grasp_dofs)

sample_grasp_q = (
  torch.cat(
    [
      grasp_pose,
      grasp_dofs,
    ]
  )
  .unsqueeze(0)
  .to(device)
  .float()
)

In [51]:

############ Grasp Transfer Process ################

grasp_transfer_opt = AdamGraspTransfer(
  source_gripper,
  target_gripper,
  learning_rate=1e-3,
  device=device
)
q_traj, energy, _ = grasp_transfer_opt.run_adam(
  sample_grasp_q.squeeze(0), running_name="test"
)
min_energy_index = energy.min(dim=0)[1]
best_q = q_traj[min_energy_index.item(), -1]
if best_q.shape[0] != 9 + len(target_model.dynamic_joints):
  # We optimized only for pose for 2-F gripper, so need to provide dummy joints
  best_q = torch.cat((best_q, (target_model.dynamic_joints_q_upper[0] - target_model.dynamic_joints_q_mid[0])), dim=0)

# Putting best_q (grasp rep), in same format as sample_grasp_q
best_q = best_q.unsqueeze(0)

In [ ]:
print(sample_grasp_q.shape, best_q.shape)

# VIZ: Hand PC, Src & Target Grippers

In [ ]:
print("Plotting TARGET and SOURCE together...")

SRC_GRP_COLOR = 'darkgray'
TRG_GRP_COLOR = '#FFAA00'
SRC_PCS_COLOR = 'lightgray'
PCS_PTS_RADIUS = 3

 
src_grp_mesh = source_model.get_plotly_data(q=sample_grasp_q, color=SRC_GRP_COLOR, opacity=0.8)
trg_grp_mesh = target_model.get_plotly_data(q=best_q.float().to(device), color=TRG_GRP_COLOR, opacity=0.5)

# Add hand point cloud saved during hamer inference
hand_ply = f"{frame_id}_{int(not use_left_hand)}.ply"
mano_mesh = trimesh.load_mesh(f"../data/{hand_ply}")
x, y, z = mano_mesh.vertices.T
src_pointcloud = [
        go.Scatter3d(
            x=x, y=y, z=z,
            mode='markers',
            marker=dict(size=PCS_PTS_RADIUS, color=SRC_PCS_COLOR)
        )
    ]

######## PLOT EVERYTHING TOGETHER ##########

vis_data = []
vis_data += src_grp_mesh
vis_data += trg_grp_mesh
vis_data += src_pointcloud

fig = go.Figure(data=vis_data)
fig.update_layout(
    scene = dict(
        xaxis = dict(visible=False),
        yaxis = dict(visible=False),
        zaxis =dict(visible=False)
        )
)
# fig.show()
fig.write_html(os.path.join(FIG_SAVE_DIR, f"viz_transfer_combined.html"))




In [ ]:
############# INDIVIDUAL PLOTS ##############

########## Just the SOURCE GRIPPER URDF MESH ############
vis_data = []
vis_data += src_grp_mesh
fig = go.Figure(data=vis_data)
fig.update_layout(
    scene = dict(
        xaxis = dict(visible=False),
        yaxis = dict(visible=False),
        zaxis =dict(visible=False)
        )
)
# fig.show()
fig.write_html(os.path.join(FIG_SAVE_DIR, f"{source_gripper}_urdf.html"))


########## Just the TARGET GRIPPER URDF MESH ############
vis_data = []
vis_data += trg_grp_mesh
fig = go.Figure(data=vis_data)
fig.update_layout(
    scene = dict(
        xaxis = dict(visible=False),
        yaxis = dict(visible=False),
        zaxis =dict(visible=False)
        )
)
# fig.show()
fig.write_html(os.path.join(FIG_SAVE_DIR, f"{target_gripper}_urdf.html"))


########## Just the SRC Hand Point Cloud ############
vis_data = []
vis_data += src_pointcloud
fig = go.Figure(data=vis_data)
fig.update_layout(
    scene = dict(
        xaxis = dict(visible=False),
        yaxis = dict(visible=False),
        zaxis =dict(visible=False)
        )
)
# fig.show()
fig.write_html(os.path.join(FIG_SAVE_DIR, f"{source_gripper}_HandPC.html"))


# VIZ: Grippers with UV Maps

In [ ]:
print("Plotting SRC GRIPPER...")

# Gripper PC + GCS U-V Color Map
src_grp_surf_pts = source_model.get_surface_points_new(sample_grasp_q).squeeze(0).cpu()
src_grp_pts_cord = source_model.get_gripper_coords().cpu()
print(src_grp_pts_cord.shape, src_grp_surf_pts.shape)

# Gripper Mesh Data
vis_data = source_model.get_plotly_data(q=sample_grasp_q, opacity=0.9)
vis_data += [
    plot_uv_map(
      src_grp_surf_pts.cpu().numpy(),
      src_grp_pts_cord.cpu().numpy(),
      size=2
    )
]
fig = go.Figure(data=vis_data)
fig.update_layout(
    scene = dict(
        xaxis = dict(visible=False),
        yaxis = dict(visible=False),
        zaxis =dict(visible=False)
        )
)
# fig.show()
fig.write_html(os.path.join(FIG_SAVE_DIR, f"{source_gripper}_UV.html"))

# fig.update_layout(template='simple_white')
# fig.update_xaxes(showgrid=False)
# fig.update_yaxes(showgrid=False)


In [ ]:
print("Plotting Target (Fetch?)...")

target_grp_surf_pts = target_model.get_surface_points_new(best_q).squeeze(0).cpu()
target_grp_pts_cord = target_model.get_gripper_coords().cpu()
print(target_grp_pts_cord.shape, target_grp_surf_pts.shape)

vis_data = target_model.get_plotly_data(q=best_q, opacity=0.9)
vis_data += [
    plot_uv_map(
      target_grp_surf_pts.cpu().numpy(),
      target_grp_pts_cord.cpu().numpy(),
      size=2,
    )
]
fig = go.Figure(data=vis_data)
fig.update_layout(
    scene = dict(
        xaxis = dict(visible=False),
        yaxis = dict(visible=False),
        zaxis =dict(visible=False)
        )
)
# fig.show()
fig.write_html(os.path.join(FIG_SAVE_DIR, f"{target_gripper}_UV.html"))

# VIZ: Sphere with UV Maps overlaid

In [75]:
sphere_mesh = trimesh.creation.uv_sphere(radius=1, count=(32,32))

In [88]:
######### VIZ SPHERE UV WITH SRC GRIPPER ##############

sp_pts_from_grp_coords = spherical_to_cart_coords(src_grp_pts_cord.cpu().numpy(), radius=1)

vis_data = [
  go.Mesh3d(
    x=sphere_mesh.vertices[:, 0],
    y=sphere_mesh.vertices[:, 1],
    z=sphere_mesh.vertices[:, 2],
    i=sphere_mesh.faces[:, 0],
    j=sphere_mesh.faces[:, 1],
    k=sphere_mesh.faces[:, 2],
    color='lightblue',
    opacity=0.3,
  )
]

vis_data += [
    plot_uv_map(
      sp_pts_from_grp_coords,
      src_grp_pts_cord.cpu().numpy(),
      size=3,
    )
]
fig = go.Figure(data=vis_data)
fig.update_layout(
    scene = dict(
        xaxis = dict(visible=False),
        yaxis = dict(visible=False),
        zaxis =dict(visible=False)
        )
)
# fig.show()
fig.write_html(os.path.join(FIG_SAVE_DIR, f"{source_gripper}_sphere_UV.html"))

In [87]:
######### VIZ SPHERE UV WITH TARGET GRIPPER ##############

sp_pts_from_grp_coords = spherical_to_cart_coords(target_grp_pts_cord.cpu().numpy(), radius=1)

vis_data = [
  go.Mesh3d(
    x=sphere_mesh.vertices[:, 0],
    y=sphere_mesh.vertices[:, 1],
    z=sphere_mesh.vertices[:, 2],
    i=sphere_mesh.faces[:, 0],
    j=sphere_mesh.faces[:, 1],
    k=sphere_mesh.faces[:, 2],
    color='lightblue',
    opacity=0.3,
  )
]

vis_data += [
    plot_uv_map(
      sp_pts_from_grp_coords,
      target_grp_pts_cord.cpu().numpy(),
      size=3,
    )
]
fig = go.Figure(data=vis_data)
fig.update_layout(
    scene = dict(
        xaxis = dict(visible=False),
        yaxis = dict(visible=False),
        zaxis =dict(visible=False)
        )
)
# fig.show()
fig.write_html(os.path.join(FIG_SAVE_DIR, f"{target_gripper}_sphere_UV.html"))